# Named Entity Recognition (NER) — Person/Organization/Location Detection

**Why this notebook exists:** your sentiment model correctly identifies emotion words, but has no
concept of *proper nouns*. `"Happy Singh is sad"` gets misread because "Happy" is treated as the
emotion word, not as someone's name. This notebook trains a **separate model** whose only job is:
given a sentence, tag EVERY WORD as one of:

- `per` — person name (e.g. "Singh", "Modi")
- `org` — organization (e.g. "Maruti Suzuki", "Reddit")
- `geo` — location (e.g. "India", "Delhi")
- `tim` — time expression (e.g. "Monday", "2024")
- `gpe` — geopolitical entity (e.g. "Indian", "American")
- `art`, `eve`, `nat` — art, event, natural-phenomenon entities (rare)
- `O` — not a named entity at all (most words)

Once trained, we use this model as a **pre-processing step** before sentiment analysis: detect
person names in the ORIGINAL sentence (before lowercasing — capitalization is a huge clue for
names), replace them with a neutral placeholder, THEN run your sentiment model. This means
`"Happy Singh is sad"` becomes `"[PERSON] is sad"` before your sentiment model ever sees the word
"Happy" as an emotion word.

**Dataset:** Kaggle "Annotated Corpus for Named Entity Recognition" (`ner_dataset.csv`),
~1 million tagged words across ~47,959 sentences.

Run every cell top to bottom.


## Step 1 — Imports

In [ ]:
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, TimeDistributed, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split

print("Libraries loaded.")


## Step 2 — Load the dataset

This file needs `encoding="latin1"` -- it has some special characters that break the default UTF-8
reader. We also forward-fill the `Sentence #` column: in the raw CSV, the sentence number is only
written on the FIRST word of each sentence and left blank for the rest, so `ffill()` fills it down.

In [ ]:
data = pd.read_csv("dataset/ner/ner_dataset.csv", encoding="latin1")
data = data.ffill()

print("Total tagged words:", len(data))
print("Unique tags:", data["Tag"].unique())
data.head(10)


## Step 3 — Exploratory Data Analysis (EDA)

How often does each entity tag appear? `O` (not-an-entity) will dominate by far -- that's normal,
most words in any sentence aren't named entities.

In [ ]:
tag_counts = data["Tag"].value_counts()
print(tag_counts)

plt.figure(figsize=(10,4))
tag_counts.plot(kind="bar", color="#7193FF")
plt.title("Entity Tag Distribution")
plt.ylabel("Count")
plt.show()


## Step 4 — Group words back into sentences

Right now the data is one row per WORD. We need to regroup it into one row per SENTENCE
(a list of words + a list of matching tags), since that's what the model actually learns from --
the sequence of words in context, not isolated words.

In [ ]:
class SentenceGetter:
    """Groups the word-by-word CSV rows back into full sentences."""
    def __init__(self, data):
        agg_func = lambda s: [
            (w, p, t) for w, p, t in zip(s["Word"].values.tolist(),
                                          s["POS"].values.tolist(),
                                          s["Tag"].values.tolist())
        ]
        grouped = data.groupby("Sentence #").apply(agg_func)
        self.sentences = list(grouped)


getter = SentenceGetter(data)
sentences = getter.sentences

print("Total sentences:", len(sentences))
print("Example sentence (word, POS tag, entity tag):")
print(sentences[0])


## Step 5 — Data Augmentation: names in more sentence positions

The base Kaggle dataset is mostly news-style text (e.g. "Barack Obama met with..."), so names
usually appear at the START of a sentence. Real conversational text often puts names in other
positions -- "my name is X", "I met X yesterday", "this is X speaking". The model under-performs
on those patterns simply because it saw very few examples of them. We fix this the same way we
fixed sentiment negation: generate synthetic tagged examples and add them to training data.

In [ ]:
import random

COMMON_NAMES = [
    "Happy Singh", "Rahul Sharma", "Priya Patel", "Amit Kumar", "Grace Thomas",
    "Hope Miller", "Joy Wilson", "John Smith", "Maria Garcia", "David Lee",
    "Neha Gupta", "Vikram Rao", "Sara Khan", "Arjun Mehta", "Anjali Verma",
]

TEMPLATES = [
    "hi my name is {name}",
    "my name is {name} and I am here",
    "this is {name} speaking",
    "I met {name} yesterday",
    "I am {name} and I feel great",
    "hello everyone I am {name}",
    "{name} said he was happy",
    "call me {name}",
    "you can reach {name} at this number",
    "{name} is sad today",
    "{name} is very happy about this",
]


def make_ner_augmented_sentences(n=800):
    """
    Generates synthetic (word, POS, tag) sentences for training, following
    the same format as the original SentenceGetter output, with the name
    words correctly tagged B-per / I-per based on their position.
    """
    synthetic_sentences = []
    for i in range(n):
        name = random.choice(COMMON_NAMES)
        template = random.choice(TEMPLATES)
        sentence_text = template.format(name=name)

        name_words = name.split()
        tokens = sentence_text.split()

        tags = []
        name_idx = 0
        j = 0
        while j < len(tokens):
            if name_idx < len(name_words) and tokens[j] == name_words[name_idx]:
                tags.append("B-per" if name_idx == 0 else "I-per")
                name_idx += 1
            else:
                tags.append("O")
            j += 1

        # POS tag doesn't matter for our model (we don't use the POS column),
        # so we just fill it with a placeholder.
        synthetic_sentences.append([(w, "NN", t) for w, t in zip(tokens, tags)])

    return synthetic_sentences


augmented_sentences = make_ner_augmented_sentences(800)
sentences = sentences + augmented_sentences
print("Total sentences after augmentation:", len(sentences))
print("Example augmented sentence:", augmented_sentences[0])


## Step 6 — Build vocabulary and convert to integer sequences

Same idea as the sentiment model's Tokenizer step: every unique word gets an integer ID, every
unique entity tag gets an integer ID, then every sentence becomes a sequence of word-IDs with a
matching sequence of tag-IDs. We keep the ORIGINAL capitalization here (unlike the sentiment
model) because capitalization is one of the strongest signals for "this is probably a name".

In [ ]:
words = list(set(w[0] for s in sentences for w in s))
words.append("ENDPAD")   # padding placeholder word
n_words = len(words)

tags = list(set(w[2] for s in sentences for w in s))
n_tags = len(tags)

word2idx = {w: i + 1 for i, w in enumerate(words)}   # 0 reserved for padding
word2idx["UNK"] = len(word2idx) + 1                  # unknown-word fallback
tag2idx = {t: i for i, t in enumerate(tags)}
idx2tag = {i: t for t, i in tag2idx.items()}

print("Vocabulary size:", n_words)
print("Number of tag types:", n_tags)
print("Tags:", tag2idx)


In [ ]:
MAX_LEN = 75   # most sentences are well under this; longer ones get truncated

X = [[word2idx.get(w[0], word2idx["UNK"]) for w in s] for s in sentences]
X = pad_sequences(maxlen=MAX_LEN, sequences=X, padding="post", value=0)

y = [[tag2idx[w[2]] for w in s] for s in sentences]
y = pad_sequences(maxlen=MAX_LEN, sequences=y, padding="post", value=tag2idx["O"])

print("X shape:", X.shape)
print("y shape:", y.shape)


## Step 7 — Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
print("Train sentences:", X_train.shape[0], " Test sentences:", X_test.shape[0])


## Step 8 — Build the BiLSTM tagging model

This is structurally similar to your sentiment LSTM, with one key difference: instead of
outputting ONE prediction for the whole sentence, `TimeDistributed(Dense(...))` outputs a
prediction for EVERY WORD position -- i.e. "is this word part of a name? an organization? nothing?"
`mask_zero=True` tells the model to ignore the padding positions rather than treating 0 as a real word.

In [ ]:
EMBEDDING_DIM = 64

input_layer = Input(shape=(MAX_LEN,))
model = Embedding(input_dim=n_words + 2, output_dim=EMBEDDING_DIM, mask_zero=True)(input_layer)
model = Bidirectional(LSTM(units=64, return_sequences=True, recurrent_dropout=0.1))(model)
output_layer = TimeDistributed(Dense(n_tags, activation="softmax"))(model)

ner_model = Model(input_layer, output_layer)
ner_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
ner_model.summary()


## Step 9 — Train

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

y_train_reshaped = np.expand_dims(y_train, -1)

history = ner_model.fit(
    X_train, y_train_reshaped,
    validation_split=0.1,
    batch_size=64,
    epochs=15,
    callbacks=[early_stop],
    verbose=1,
)


## Step 10 — Evaluate on the held-out test set

In [ ]:
y_test_reshaped = np.expand_dims(y_test, -1)
test_loss, test_accuracy = ner_model.evaluate(X_test, y_test_reshaped, verbose=0)
print("Test accuracy (per-word tag accuracy, including padding positions):", round(test_accuracy, 4))


## Step 11 — Sanity check on real sentences

The actual test that matters: does it correctly find the person name in the exact sentences
that were breaking your sentiment model?

In [ ]:
def predict_entities(sentence: str):
    """
    Takes a raw sentence (original capitalization, NOT lowercased), tags each
    word, and returns a list of (word, predicted_tag) pairs.
    """
    tokens = sentence.split()
    seq = [word2idx.get(w, word2idx["UNK"]) for w in tokens]
    padded = pad_sequences([seq], maxlen=MAX_LEN, padding="post", value=0)

    pred = ner_model.predict(padded, verbose=0)[0]
    pred_tags = [idx2tag[i] for i in np.argmax(pred, axis=-1)][:len(tokens)]

    return list(zip(tokens, pred_tags))


test_sentences = [
    "Happy Singh is sad",
    "Maruti Suzuki India Limited is based in Delhi",
    "I met Rahul yesterday in Mumbai",
    "This is not good at all",
]

for s in test_sentences:
    print(s, "-->", predict_entities(s))
    print()


## Step 12 — Save all artifacts

These 4 files are what `utils/ner.py` in your Streamlit app will load. Note these are SEPARATE
from your sentiment model files -- this is a second, independent model.

In [ ]:
ner_model.save("ner_model.keras")

with open("ner_word2idx.pkl", "wb") as f:
    pickle.dump(word2idx, f)

with open("ner_idx2tag.pkl", "wb") as f:
    pickle.dump(idx2tag, f)

ner_config = {
    "max_len": MAX_LEN,
    "unk_index": word2idx["UNK"],
    "test_accuracy": float(test_accuracy),
}
with open("ner_config.json", "w") as f:
    json.dump(ner_config, f)

print("Saved: ner_model.keras, ner_word2idx.pkl, ner_idx2tag.pkl, ner_config.json")
print("Test accuracy:", round(test_accuracy, 4))


## Next step

Once this finishes and you have the 4 new files (`ner_model.keras`, `ner_word2idx.pkl`,
`ner_idx2tag.pkl`, `ner_config.json`) sitting next to this notebook:

1. Copy all 4 files into your project's `models/` folder (alongside the sentiment `_v2` files --
   they won't conflict, different filenames).
2. Ask me for `utils/ner.py` and the updated `utils/predictor.py` -- these will run NER on the
   raw sentence FIRST (before lowercasing), mask out any detected person names, THEN run your
   sentiment model on the masked text.
3. Restart the Streamlit app and re-test "Happy Singh is sad".
